# Rapport de risque

Ce rapport lit `data/07_risk/`. Il ne réestime pas les poids. La page est `reports/risk/dashboard.html`. Elle est réécrite par `build_risk` à chaque reconstruction. La méthode, avec les conventions d'unité, de signe et d'échantillon, est `documentation/08_risk_analytics.ipynb`. Le détail des paires et des secteurs est `notebooks/08_analyse_risque.ipynb`.

La cellule imprime le seuil lu dans le module, le rang $k$, puis pour chaque livre la volatilité, la VaR, l'expected shortfall et les mois de la queue. Elle compare ensuite la volatilité la plus basse et la VaR la plus profonde, à partir du fichier.

La corrélation moyenne décrit toutes les paires. La cellule nomme le secteur le plus lié parmi ceux qui ont au moins deux paires, et elle compte les paires au-dessus du seuil de lecture. Les chiffres viennent des fichiers de l'exécution.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

import math

from src.risk.build_risk import METRICS_OUTPUT, SUMMARY_FILE, TAIL_FILE, VAR_LEVEL

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}


def percent(value):
    return f"{float(value) * 100:.1f}".replace(".", ",") + " %"


metrics = pd.read_csv(METRICS_OUTPUT)
tail = pd.read_csv(TAIL_FILE)
summary = pd.read_csv(SUMMARY_FILE).iloc[0]
start = metrics["window_start"].iloc[0]
end = metrics["window_end"].iloc[0]
months = int(metrics["months"].iloc[0])
print(f"Période {start} – {end}, {months} mois.")
rank = math.ceil(VAR_LEVEL * months)
print(
    f"Seuil {percent(1 - VAR_LEVEL)}. "
    f"Rang retenu : {rank}, soit le plafond de {percent(VAR_LEVEL)} des {months} mois."
)
print(
    f"Corrélation moyenne des {int(summary['names'])} titres inclus : "
    f"{percent(summary['average_correlation'])}, hors diagonale."
)
print("Cette moyenne décrit la liste partagée. Elle est la même pour les trois livres.")
print()
for strategy, label in LABELS.items():
    row = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    months_tail = tail.loc[tail["strategy"].eq(strategy)].sort_values("rank")
    listed = ", ".join(
        f"{item.month} ({percent(item.portfolio_return)})"
        for item in months_tail.itertuples(index=False)
    )
    print(label)
    print(f"Volatilité annualisée {percent(row['annualized_volatility'])}.")
    print(
        f"VaR 95 % {percent(row['var_95'])}. "
        f"Expected shortfall 95 % {percent(row['expected_shortfall_95'])}."
    )
    print(f"Mois de la queue : {listed}.")
    print()
equal = float(metrics.loc[metrics["strategy"].eq("equal_weight"), "var_95"].iloc[0])
quiet = float(metrics.loc[metrics["strategy"].eq("minimum_volatility"), "var_95"].iloc[0])
if quiet < equal:
    print(
        "La volatilité minimale a une VaR historique plus profonde que l'équipondéré. "
        "Le livre le moins variable sur toute la période n'a pas la queue mensuelle la plus douce."
    )

matrix = pd.read_csv(PROJECT_ROOT / "data/07_risk/correlation_matrix.csv", index_col="symbol")
membership = pd.read_csv(PROJECT_ROOT / "data/04_portfolios/portfolio_membership.csv")
from src.portfolio_monitoring.build_monitoring import load_holding_classification
from src.risk.build_risk import PAIR_LIST_FLOOR

included = membership.loc[membership["decision"].eq("included"), ["yahoo_symbol", "ticker"]]
identity = included.merge(
    load_holding_classification()[["ticker", "sector"]],
    on="ticker",
    how="left",
)
sector_of = identity.set_index("yahoo_symbol")["sector"]
symbols = list(matrix.columns)
values = []
same_sector = {}
cross = []
strong_by_sector = {}
for i, left in enumerate(symbols):
    for right in symbols[i + 1 :]:
        value = float(matrix.loc[left, right])
        values.append(value)
        left_sector = str(sector_of.loc[left])
        right_sector = str(sector_of.loc[right])
        if left_sector == right_sector:
            same_sector.setdefault(left_sector, []).append(value)
            if value >= PAIR_LIST_FLOOR:
                strong_by_sector[left_sector] = strong_by_sector.get(left_sector, 0) + 1
        else:
            cross.append(value)
            if value >= PAIR_LIST_FLOOR:
                strong_by_sector["deux secteurs"] = strong_by_sector.get("deux secteurs", 0) + 1
linked = {sector: rows for sector, rows in same_sector.items() if len(rows) >= 2}
tight_sector = max(linked, key=lambda sector: sum(linked[sector]) / len(linked[sector]))
print(
    f"Paires : {len(values)}. Médiane {percent(pd.Series(values).median())}. "
    f"Paires d'au moins {percent(PAIR_LIST_FLOOR)} : {sum(strong_by_sector.values())}."
)
for sector, count in sorted(strong_by_sector.items(), key=lambda item: (-item[1], item[0])):
    print(f"  {sector} : {count}.")
print(
    f"Secteur le plus lié, parmi ceux qui ont au moins deux paires : {tight_sector}, "
    f"moyenne {percent(sum(linked[tight_sector]) / len(linked[tight_sector]))}."
)
print(
    f"Entre deux secteurs différents : {percent(sum(cross) / len(cross))}."
)
print("Le détail des noms et des poids est dans le notebook d'analyse.")


Période 2016-11 – 2026-10, 120 mois.
Seuil 95,0 %. Rang retenu : 6, soit le plafond de 5,0 % des 120 mois.
Corrélation moyenne des 28 titres inclus : 16,3 %, hors diagonale.
Cette moyenne décrit la liste partagée. Elle est la même pour les trois livres.

Équipondéré
Volatilité annualisée 12,3 %.
VaR 95 % -5,5 %. Expected shortfall 95 % -7,9 %.
Mois de la queue : 2020-03 (-11,9 %), 2022-09 (-10,1 %), 2020-10 (-7,4 %), 2018-10 (-6,5 %), 2020-02 (-5,7 %), 2022-01 (-5,5 %).

Volatilité minimale
Volatilité annualisée 11,8 %.
VaR 95 % -5,6 %. Expected shortfall 95 % -7,6 %.
Mois de la queue : 2020-03 (-11,3 %), 2022-09 (-9,2 %), 2020-10 (-6,6 %), 2022-01 (-6,6 %), 2020-02 (-6,4 %), 2018-10 (-5,6 %).

Sharpe maximal
Volatilité annualisée 12,1 %.
VaR 95 % -5,9 %. Expected shortfall 95 % -8,2 %.
Mois de la queue : 2022-09 (-11,9 %), 2020-03 (-11,2 %), 2020-10 (-7,6 %), 2022-01 (-6,3 %), 2020-02 (-6,2 %), 2023-10 (-5,9 %).

La volatilité minimale a une VaR historique plus profonde que l'équipond

Paires : 378. Médiane 14,5 %. Paires d'au moins 50,0 % : 13.
  Real Estate : 12.
  Health Care : 1.
Secteur le plus lié, parmi ceux qui ont au moins deux paires : Real Estate, moyenne 36,7 %.
Entre deux secteurs différents : 13,6 %.
Le détail des noms et des poids est dans le notebook d'analyse.
